## Predicting Match Outcomes with Dixon-Coles and Monte Carlo Simulation

### Model Choice 
---
> **Why Dixon-Coles + Monte Carlo?**

>  Standard Poisson regression underestimates low-scoring results like `0-0`, `1-0`, and `1-1`, which are extremely common in international football. The **Dixon-Coles** model corrects this with a bivariate adjustment for these scorelines. Match outcome probabilities are then fed into a **Monte Carlo simulation**, running the full 2026 tournament bracket **10,000 times** to produce win probabilities for each team rather than a single deterministic prediction.
---

### Why not just Poisson?
##### - Poisson treats home and away goals as independent.
##### - It consistently underpredicts draws and narrow wins.
##### - Dixon-Coles fixes this with a small but impactful correction factor `ρ (rho)`.

### Why Monte Carlo?
##### - A single simulation gives one outcome, not very useful.
##### - 10,000 runs gives a *probability distribution* across all teams.
##### - Much more honest about uncertainty in tournament prediction.

In [16]:
# Importing the necessary Libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import poisson

In [17]:
# Loading the Team Strength DataFrame 

base_dir = Path.cwd()
while base_dir.name != "World Cup Predictor":
    base_dir = base_dir.parent
ts_path = base_dir/"data"/"processed"/"team_strength.csv"
ts_df = pd.read_csv(ts_path)

## Dixon-Coles Correction

---

> Standard Poisson treats home and away goals as fully independent.

> **Dixon-Coles** introduces a correction factor **τ (tau)** for four low-scoring outcomes that Poisson consistently mispredicts.

---

### The 4 corrected scorelines

| Scoreline | τ (tau) |
|-----------|---------|
| `0 - 0`   | `1 - λ_home × λ_away × ρ` |
| `1 - 0`   | `1 + λ_away × ρ` |
| `0 - 1`   | `1 + λ_home × ρ` |
| `1 - 1`   | `1 - ρ` |
| all others | `1` (no correction) |

### What is ρ (rho)?
- A small **negative** correlation parameter, typically set to `-0.1`.
- Represents the slight negative correlation between both teams' scores in tight games.
- All other scorelines remain pure Poisson. Only these 4 are adjusted.

In [18]:
# Dixon-Coles Correction Function

def dixon_coles_correction(home_goals, away_goals, lambda_home, lambda_away, rho=-0.1):
    if home_goals == 0 and away_goals == 0:
        tau = 1 - lambda_home*lambda_away*rho
    elif home_goals == 1 and away_goals == 0:
        tau = 1 + lambda_away*rho
    elif home_goals == 0 and away_goals == 1:
        tau = 1 + lambda_home*rho
    elif home_goals == 1 and away_goals == 1:
        tau = 1 - rho
    else:
        tau = 1

    return tau

In [19]:
# The main function that will be predicting the match outcomes

def predict_outcome(home_team: str, away_team: str):
    '''
    ts_df.at[row, column] is just pandas for "give me the value at this specific row and column", like looking up a cell in a table.
    So:
      - team_strength_df.at['Brazil', 'goals_for'] → average goals Brazil scores
      - team_strength_df.at['France', 'goals_against'] → average goals France concedes

    When you multiply them:
      - High attack × High defense conceded = high lambda (expect lots of goals)
      - Low attack × Low defense conceded = low lambda (expect few goals)
    
    In the function, home_team and away_team are the row names, and 'goals_for'/'goals_against' are the column names.
    '''
    lambda_home = ts_df.at[home_team, 'goals_for'] * ts_df.at[away_team, 'goals_against']
    lambda_away = ts_df.at[away_team, 'goals_for'] * ts_df.at[home_team, 'goals_against']

    # Double looping over all scorelines from 0-10
    prob_home, prob_away, prob_draw = 0,0,0
    for i in range(11):
        for j in range(11):
            base_probability = poisson.pmf(i, lambda_home) * poisson.pmf(j, lambda_away)
            dc_correction = dixon_coles_correction(i, j, lambda_home, lambda_away)
            corrected_probability = base_probability*dc_correction
            if i>j:
                prob_home += corrected_probability
            elif j>i:
                prob_away += corrected_probability
            else:
                prob_draw += corrected_probability

    return (prob_home, prob_away, prob_draw)

In [23]:
prob_home, prob_away, prob_draw = predict_outcome('Brazil','France')
prob_sum = prob_home+prob_away+prob_draw
print(prob_sum)

0.9999864172882194
